# 2.2 — Train a Policy on the GPU with mjlab

Trains directional myoLeg locomotion (`myoLegDirectionalForward-v0`) with thousands of parallel envs on mjlab (MuJoCo Warp + RSL-RL), then plays the policy back on the CPU env.

MyoSuite tasks have two matched halves under one `env_id`:
- **CPU** (`gym.make`, MuJoCo C++): playback, fine-tuning, debugging.
- **GPU** (mjlab): thousands of envs in parallel for fast RL.

Observation (153-d) and muscle action space are identical, so a policy **trained on GPU transfers to CPU** unchanged.

**Prerequisites:** Completed notebook 1.1 · CPU parts run anywhere with `pip install -e .`; training needs Linux + CUDA and `pip install -e ".[mjlab]"`.

The helper script is `files/2.2/directional_leg_gpu_training.py`.

In [ ]:
ENV_ID = "myoLegDirectionalForward-v0"

### 2.2.1 CPU: check the env and roll out a random policy

In [ ]:
!python files/2.2/directional_leg_gpu_training.py --cpu-demo

### 2.2.2 GPU: train

A short smoke run (needs CUDA); for a real run use the training CLI with many envs, e.g.
`python scripts/train_mjlab.py myoLegDirectionalForward-v0 --env.scene.num-envs 1024`.

In [ ]:
!python files/2.2/directional_leg_gpu_training.py --gpu-train --iterations 5

### 2.2.3 CPU: play back the trained checkpoint

In [ ]:
from pathlib import Path

import gymnasium as gym
import numpy as np

import myosuite  # noqa: F401  (registers the envs)
from myosuite.utils.checkpoint_utils import find_checkpoint, load_policy

env = gym.make(ENV_ID)
act = load_policy(env, find_checkpoint(ENV_ID, roots=(Path.cwd(), Path.cwd().parent)))  # newest mjlab run, else random

obs, _ = env.reset(seed=0)
total = 0.0
for _ in range(200):
    obs, reward, terminated, truncated, info = env.step(act(obs))
    total += float(reward)
    if terminated or truncated:
        obs, _ = env.reset()
env.close()
print(f"return over 200 steps: {total:.2f}")